
# Assignment 4: Embedding Models, Dense Retrieval, and RAG

**Student names**: Kirsten Sunde Thaulow, Nora Marie Thon Børaas <br>
**Group number**: 3 <br>
**Date**: 26.10.2025

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 26.10.2025 23:59

In this assignment, you will:
- Build a vector search index over a blog corpus using sentence embeddings
- Implement dense retrieval (cosine similarity)
- Use the vector index as the foundation for a simple Retrieval-Augmented Generation (RAG) chat system with evaluation on three queries



---
## Dataset

You will use the blog files, provided in the folder: 
- `blogs-sample` (in the same directory as this notebook)

Use only the blog files provided in the folder below. Each file contains multiple `<post>` elements. Treat **each `<post>` as a separate document**.

**The code to parse files is not provided. Implement the loading yourself in 4.1.**



## 4.1 – Load and parse blog documents

Load all XML files from `blogs-sample`, extract the text of each `<post>`, and store one string per document. Keep the raw text per post as the document text.

You may experience some trouble parsing all lines in the files, but this is okay.



In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

POST_RE = re.compile(r"<post>(.*?)</post>", flags=re.S | re.I)

def load_posts(folder="blogs-sample"):
    docs = []
    for xmlfile in Path(folder).glob("*.xml"):
        try:
            root = ET.parse(xmlfile).getroot()
            for post in root.findall(".//post"):
                text = "".join(post.itertext()).strip()
                if text:
                    docs.append(text)
        except ET.ParseError:
            # Fallback: regex-grab posts if XML is slightly broken
            txt = xmlfile.read_text(encoding="utf-8", errors="ignore")
            for m in POST_RE.findall(txt):
                cleaned = " ".join(m.split()).strip()
                if cleaned:
                    docs.append(cleaned)
    return docs

documents = load_posts("blogs-sample")
print("Loaded:", len(documents), "posts")
print(documents[0][:200])


Loaded: 5231 posts
About to go t bed late (again) got sucked into (another) late night film. Tonight was urlLink Maybe Baby . It was really good made me think, but not about babies. The guy screws up his marriage and it



## 4.2 – Embedding Models

Select and load a sentence embedding model (e.g., `sentence-transformers/all-MiniLM-L6-v2`) and compute embeddings for all documents.

- Store document embeddings in a variable named `doc_embeddings`.
- Ensure that the same model will be used for query encoding later.

**Report**:
- The embedding matrix shape 


In [2]:

# TODO: Load a sentence embedding model and encode all documents into `doc_embeddings`.
# You may use `sentence-transformers`. Report the embedding matrix shape.

# Your code here
from sentence_transformers import SentenceTransformer
import numpy as np

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

doc_embeddings = embedder.encode(
    documents,
    batch_size = 32,
    convert_to_numpy = True, 
    show_progress_bar = True
)

doc_embeddings = np.asarray(doc_embeddings)
print("Embedding matrix shape:", doc_embeddings.shape)



c:\Users\noram\OneDrive - NTNU\INFORMASJONSGJENFINNING\information-retrieval-assignment-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 164/164 [00:48<00:00,  3.35it/s]


Embedding matrix shape: (5231, 384)



## 4.3 – Dense Retrieval

Implement a cosine similarity search over `doc_embeddings` for a given query.

- Write a function `dense_search(query: str, k: int = 5) -> list[int]` that returns the indices of the top-k documents.
- Use the same embedding model to encode the query.
- Use cosine similarity for ranking.

**Report**:
- Results for the provided query showing the indices of the top results.


In [3]:

# TODO: Implement dense retrieval using cosine similarity.
# Function signature to implement:
# def dense_search(query: str, k: int = 5) -> list[int]:

# Your code here
doc_matrix = np.asarray(doc_embeddings, dtype=np.float32)
doc_norms = np.linalg.norm(doc_matrix, axis=1, keepdims=True) + 1e-12
doc_matrix_norm = doc_matrix / doc_norms

def dense_search(query: str, k: int = 5) -> list[int]:
    query_vec = embedder.encode([query], convert_to_numpy=True)[0]
    query_vec /= (np.linalg.norm(query_vec) + 1e-12)
    sims = doc_matrix_norm @ query_vec  # cosine similarities
    topk = np.argsort(-sims)[:k]
    return topk.tolist()



#Report
print(dense_search("How do people feel about their jobs?", k=5))


[651, 62, 1706, 4951, 2024]



## 4.4 – Build a Vector Search Index

Build a lightweight vector index structure to enable repeated querying efficiently.

- You may reuse `doc_embeddings` directly or create an index structure. Ensure the index can return top-k document indices given a query vector.


In [4]:

# TODO: Initialize a vector index over `doc_embeddings`
# Keep code minimal. The goal is to enable fast top-k retrieval for repeated queries.

# Your code here   

# Chose to use flat index, because there are only 5231 vectors, so there is no point in using ANNS 

import faiss

dimension = doc_matrix.shape[1]
faiss.normalize_L2(doc_matrix)

index = faiss.IndexFlatIP(dimension)
index.add(doc_matrix)

def vector_search_index(query: str, k: int):
    query_vec = embedder.encode([query], convert_to_numpy=True)[0].astype(np.float32)
    query_vec = query_vec.reshape(1, -1)
    faiss.normalize_L2(query_vec)

    distances, indices = index.search(query_vec, k)

    return indices[0].tolist()

print(vector_search_index("How do people feel about their jobs?", k=5))


[651, 62, 1706, 4951, 2024]



## 4.5 – RAG (Retrieval-Augmented Generation)

Implement a simple RAG pipeline that:
1) Retrieves the top-k documents for a user query using your vector index.
2) Builds a prompt that includes the query and the retrieved document snippets.
3) Uses a text generation model (your choice) to produce an answer grounded in the retrieved snippets.

- Implement a function `rag_answer(query: str, k: int = 5) -> str`.
- Keep the prompt simple and state clearly that the model should rely on the provided context.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, re

model_name = "EleutherAI/gpt-neo-1.3B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.eos_token_id
torch.set_num_threads(4)

PER_DOC_CHARS = 180
MAX_CONTEXT_CHARS = 700
MAX_NEW_TOKENS = 96

def _trunc(s: str, n: int) -> str:
    if len(s) <= n: return s
    cut = s.rfind(".", 0, n)
    if cut == -1: cut = n
    return s[:cut].rstrip() + " …"

# Previous tries, had very many repetitions of the same sentences, this function is trying to remove that
def _dedupe_sentences(text: str) -> str:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    seen, out = set(), []
    for p in parts:
        key = p.lower().strip()
        if key and key not in seen:
            seen.add(key)
            out.append(p)
    return " ".join(out)

def rag_answer(query: str, k: int = 5) -> str:
    indices = vector_search_index(query, k=k)

    snippets = [_trunc(documents[i], PER_DOC_CHARS) for i in indices]
    context = "\n\n".join(snippets)
    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS] + " …"

    prompt = (
        "Answer ONLY using the SOURCE CONTEXT. "
        "If the answer is not in the context, say you don't know from the sources. "
        "Write one concise paragraph (2–3 sentences). "
        "Avoid repeating phrases. Cite sources as [#].\n\n"
        "SOURCE CONTEXT:\n" +
        "\n".join(f"[{j+1}] {t}" for j, t in enumerate(snippets)) +
        f"\n\nQUESTION:\n{query}\n\nANSWER:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        no_repeat_ngram_size=4,
        repetition_penalty=1.5,
        length_penalty=0.0,
        eos_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    answer = text[len(prompt):].strip()
    return _dedupe_sentences(answer)

Found it very difficult to find the right model, which gave satisfactory answers. Many of the models didn't give any answers, and some used way too long time. The results are a bit limited by being computed on a regular laptop.

## 4.6 – Evaluation

Use the following queries for your evaluation. For each query:

- Run `dense_search(query, k=5)` to retrieve relevant documents.
- Use `rag_answer(query, k=5)` to generate an answer using the top-5 retrieved documents.

**Queries:**
1. How do people deal with breakups?
2. What do bloggers write about their daily routines?
3. How do people feel about their jobs?


In [9]:
# Do not change this code
queries = [
    "How do people deal with breakups?",
    "What do bloggers write about their daily routines?",
    "How do people feel about their jobs?"
]

In [16]:
# TODO: Run and report your evaluation as described above.

def run_batch_evaluation(queries, k=5):
    for i, query in enumerate(queries, 1):
        print("=" * 100)
        print(f"Q{i}: {query}")
        print("-" * 100)

        top_k = dense_search(query, k=k)
        print(f"Top-{k} retrieved indices:", top_k)
        print("\nTop retrieved snippets:")
        for idx in top_k:
            snippet = documents[idx].replace("\n", " ").strip()
            print(f"[{idx}] {snippet[:200]}...\n")

        print("RAG answer:\n")
        answer = rag_answer(query, k=k)
        print(answer)
        print("\n")

# Run the evaluation
run_batch_evaluation(queries, k=5)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Q1: How do people deal with breakups?
----------------------------------------------------------------------------------------------------
Top-5 retrieved indices: [2566, 4816, 2682, 3664, 4788]

Top retrieved snippets:
[2566] Why do my ex-boyfriends always feel the need to keep in touch? Does this happen to anyone else? Particularly perplexing are the calls and emails from the ones who dumped me...didn't they want out in t...

[4816] i've got what i want... she wont call or contact me anymore... so the games finally over and its time to rebuild, recoup, find happiness alone, and then venture off into the world again as a stronger ...

[2682] Well, it looks as though I'm a single man again. Joe got back from Spain...and rather than me launching into the "we need to talk" speech...he did. Which, is really quite surprising given his continua...

[3664] Find the new. To no one in particular: Sometimes it's too easy to get stuck in the past, things that were done wrong to you. Shit happens

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


I think the best way to handle a breakup is to be honest with yourself. You can't change the past, but you can change your attitude. You can learn from the past and move on. If you're going to be honest, you should tell someone about it. If you're going through a breakup, you should probably tell someone. If you don't tell someone, you'll be embarrassed and ashamed. You can't change what happened,


Q2: What do bloggers write about their daily routines?
----------------------------------------------------------------------------------------------------
Top-5 retrieved indices: [2815, 2813, 964, 3332, 5124]

Top retrieved snippets:
[2815] From Internet Studies: Weblogging 101 For those of you who may be new to weblogging...or "blogging" as we call it, here are a few tips which will help you not look and sound completely silly when disc...

[2813] So, I posted on the Blogger discussion area that I am interested in participating in a group blog...essentially, a blog written and maintained

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


I think that blogging is a great way to share your thoughts and experiences with others. I have found that blogging is very useful for keeping in touch with friends and family. I also find that blogging is fun and interesting. QUESTIONS:
What are some of the things that bloggers write about? answers:
I have found that blogs are very useful for sharing my thoughts and experiences. I have also found that blogging can be fun and interesting,


Q3: How do people feel about their jobs?
----------------------------------------------------------------------------------------------------
Top-5 retrieved indices: [651, 62, 1706, 4951, 2024]

Top retrieved snippets:
[651] urlLink Workplace : "For INFPs the job search can be an opportunity to use their creativity, flexibility and their skills in self-expression. They can generate a variety of job possibilities, consider...

[62] My Real Job It is frustrating, this waiting, this wondering about what exactly it is that I am supposed to be doing her